In [5]:
import joblib
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import files
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential


def pipeline_com_delta_energia():
  # 1. Upload e extração da árvore JSON do Firebase Realtime Database
  print("Faça o upload do arquivo JSON exportado do Firebase:")
  uploaded = files.upload()
  if not uploaded:
    print("Nenhum arquivo selecionado.")
    return

  nome_arquivo = list(uploaded.keys())[0]
  with open(nome_arquivo, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

  # Navega pela estrutura: leituras -> data -> hora -> métricas
  leituras_dict = raw_data.get("leituras", raw_data)
  registros = []

  for data_str, horas_dict in leituras_dict.items():
    if not isinstance(horas_dict, dict):
      continue
    for hora_str, valores in horas_dict.items():
      if isinstance(valores, dict):
        linha = {"data": data_str, "hora": hora_str}
        linha.update(valores)
        registros.append(linha)

  df = pd.DataFrame(registros)

  # Ordena cronologicamente por data e hora
  df = df.sort_values(by=["data", "hora"]).reset_index(drop=True)

  print(f"\nArquivo '{nome_arquivo}' processado com sucesso!")
  print(f"Total de registros extraídos: {len(df)}")
  print(f"Colunas disponíveis: {df.columns.tolist()}\n")

  # 2. Definição das Variáveis e Cálculo do Delta de Energia
  # Usa 'potencia' como alvo principal (ou altere para 'consumo_diario' / 'delta_energia' se preferir)
  coluna_alvo = "delta_energia"

  # Calcula o delta de energia entre leituras consecutivas do medidor PZEM
  if "consumo_total_pzem" in df.columns:
    delta = df["consumo_total_pzem"].diff().fillna(0)
    # Zera deltas negativos caso o medidor PZEM tenha sido resetado em algum momento
    df["delta_energia"] = delta.clip(lower=0)
  else:
    df["delta_energia"] = df[coluna_alvo].diff().fillna(0).clip(lower=0)

  # Garante que todas as colunas de sensores sejam numéricas e sem valores nulos
  colunas_possiveis = [
      "potencia",
      "corrente",
      "voltagem",
      "consumo_diario",
      "consumo_total_pzem",
      "delta_energia",
  ]
  recursos_multivariados = [c for c in colunas_possiveis if c in df.columns]

  for col in recursos_multivariados:
    df[col] = pd.to_numeric(df[col], errors="coerce")
  df = df.dropna(subset=recursos_multivariados).reset_index(drop=True)

  print(f"Coluna Alvo: {coluna_alvo}")
  print(f"Recursos Multivariados utilizados: {recursos_multivariados}\n")

  dados_x = df[recursos_multivariados].values
  dados_y = df[[coluna_alvo]].values

  # 3. Normalização (MinMaxScaler entre 0 e 1)
  scaler_x = MinMaxScaler(feature_range=(0, 1))
  scaler_y = MinMaxScaler(feature_range=(0, 1))

  dados_x_scaled = scaler_x.fit_transform(dados_x)
  dados_y_scaled = scaler_y.fit_transform(dados_y)

  # 4. Criação das Janelas Temporais (Look-back)
  look_back = 10  # 10 leituras anteriores (~2h30 considerando intervalos de 15 min)
  X, y = [], []
  for i in range(look_back, len(dados_x_scaled)):
    X.append(dados_x_scaled[i - look_back : i])
    y.append(dados_y_scaled[i, 0])

  X, y = np.array(X), np.array(y)

  tamanho_treino = int(len(X) * 0.8)
  X_train, X_test = X[:tamanho_treino], X[tamanho_treino:]
  y_train, y_test = y[:tamanho_treino], y[tamanho_treino:]

  # 5. Construção e Treinamento do Modelo LSTM Multivariada
  print("Treinando o modelo LSTM Multivariada (Conforme Seção 4.2.1)...")
  model_lstm = Sequential([
      Input(shape=(look_back, len(recursos_multivariados))),
      LSTM(50, return_sequences=True),
      Dropout(0.2),
      LSTM(50, return_sequences=False),
      Dropout(0.2),
      Dense(25, activation="relu"),
      Dense(1),
  ])
  model_lstm.compile(optimizer="adam", loss="mean_squared_error")

  # Executado com 50 épocas exatas conforme a dissertação
  model_lstm.fit(X_train, y_train, epochs=50, batch_size=16, verbose=1)

  # 6. Avaliação no conjunto de teste
  pred_lstm_scaled = model_lstm.predict(X_test)
  pred_lstm = scaler_y.inverse_transform(pred_lstm_scaled)
  y_test_real = scaler_y.inverse_transform(y_test.reshape(-1, 1))

  mae = mean_absolute_error(y_test_real, pred_lstm)
  rmse = np.sqrt(mean_squared_error(y_test_real, pred_lstm))
  print(f"\nMétricas no Teste -> MAE: {mae:.4f} | RMSE: {rmse:.4f}")

  # 7. Salvamento do Modelo e dos Scalers
  print("\nSalvando modelo e scalers no ambiente do Colab...")
  model_lstm.save("modelo_lstm_multivariado.keras")
  joblib.dump(scaler_x, "scaler_x.pkl")
  joblib.dump(scaler_y, "scaler_y.pkl")
  print("Arquivos salvos com sucesso!")

  files.download("modelo_lstm_multivariado.keras")
  files.download("scaler_x.pkl")
  files.download("scaler_y.pkl")


# Executa o pipeline completo
pipeline_com_delta_energia()

Faça o upload do arquivo JSON exportado do Firebase:


Saving consumo-conciente-default-rtdb-export.json to consumo-conciente-default-rtdb-export (3).json

Arquivo 'consumo-conciente-default-rtdb-export (3).json' processado com sucesso!
Total de registros extraídos: 7234
Colunas disponíveis: ['data', 'hora', 'consumo_diario', 'consumo_total_pzem', 'corrente', 'potencia', 'voltagem']

Coluna Alvo: delta_energia
Recursos Multivariados utilizados: ['potencia', 'corrente', 'voltagem', 'consumo_diario', 'consumo_total_pzem', 'delta_energia']

Treinando o modelo LSTM Multivariada (Conforme Seção 4.2.1)...
Epoch 1/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 0.0015
Epoch 2/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 7.8881e-04
Epoch 3/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 5.7222e-04
Epoch 4/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 5.1117e-04
Epoch 5/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 4.2003e-04
Epoch 6/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 3.8844e-04
Epoch 7/50
362/362 ━

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>